In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_keyword_master = (
    spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Bronze.Lakehouse/Files/bronze/keyword_master/keyword_master.csv")
)

print("Rows:", df_keyword_master.count())
print("Columns:", len(df_keyword_master.columns))

display(df_keyword_master.limit(10))


StatementMeta(, b8966820-da8b-4ece-871b-fbff0f3312c1, 3, Finished, Available, Finished, False)

Rows: 2000
Columns: 8


SynapseWidget(Synapse.DataFrame, 20a75876-1dc0-4a9f-badd-996db7e70519)

In [2]:
duplicate_keyword_ids = (
    df_keyword_master
    .groupBy("keyword_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate keyword IDs:",
    duplicate_keyword_ids.count()
)

display(
    duplicate_keyword_ids
    .orderBy(F.col("count").desc())
    .limit(10)
)

StatementMeta(, b8966820-da8b-4ece-871b-fbff0f3312c1, 4, Finished, Available, Finished, False)

Duplicate keyword IDs: 0


SynapseWidget(Synapse.DataFrame, c235950a-eca7-4644-9b5e-8ebeae04e00d)

In [3]:
print("Keyword types:")

display(
    df_keyword_master
    .groupBy("keyword_type")
    .count()
    .orderBy("keyword_type")
)

StatementMeta(, b8966820-da8b-4ece-871b-fbff0f3312c1, 5, Finished, Available, Finished, False)

Keyword types:


SynapseWidget(Synapse.DataFrame, 9ee53d21-30c0-4a53-978d-8504bae1c41d)

In [4]:
print("Search intent:")

display(
    df_keyword_master
    .groupBy("search_intent")
    .count()
    .orderBy("search_intent")
)

StatementMeta(, b8966820-da8b-4ece-871b-fbff0f3312c1, 6, Finished, Available, Finished, False)

Search intent:


SynapseWidget(Synapse.DataFrame, 8374f2f6-228d-489e-9154-82b5490fe303)

In [5]:
print("Active flag:")

display(
    df_keyword_master
    .groupBy("active_flag")
    .count()
    .orderBy("active_flag")
)

StatementMeta(, b8966820-da8b-4ece-871b-fbff0f3312c1, 7, Finished, Available, Finished, False)

Active flag:


SynapseWidget(Synapse.DataFrame, 19d72569-050e-4011-9995-127f74711f83)

In [6]:
print("Null counts:")

for column_name in [
    "keyword_id",
    "keyword",
    "keyword_type",
    "category",
    "search_intent",
    "search_engine"
]:
    null_count = (
        df_keyword_master
        .filter(F.col(column_name).isNull())
        .count()
    )

    print(f"{column_name}: {null_count}")

StatementMeta(, b8966820-da8b-4ece-871b-fbff0f3312c1, 8, Finished, Available, Finished, False)

Null counts:
keyword_id: 0
keyword: 0
keyword_type: 0
category: 0
search_intent: 0
search_engine: 0


In [7]:
df_keyword_dim = (
    df_keyword_master
    .withColumn("keyword_id", F.upper(F.trim(F.col("keyword_id"))))
    .withColumn("keyword", F.trim(F.col("keyword")))
    .withColumn("keyword_type", F.initcap(F.trim(F.col("keyword_type"))))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("search_intent", F.initcap(F.trim(F.col("search_intent"))))
    .withColumn("search_engine", F.initcap(F.trim(F.col("search_engine"))))
    .withColumn("active_flag", F.trim(F.col("active_flag")))
)

StatementMeta(, b8966820-da8b-4ece-871b-fbff0f3312c1, 9, Finished, Available, Finished, False)

In [8]:
print("Keyword types:")

display(
    df_keyword_dim
    .groupBy("keyword_type")
    .count()
    .orderBy("keyword_type")
)

print("Search intent:")

display(
    df_keyword_dim
    .groupBy("search_intent")
    .count()
    .orderBy("search_intent")
)

print("Active flag:")

display(
    df_keyword_dim
    .groupBy("active_flag")
    .count()
    .orderBy("active_flag")
)

StatementMeta(, b8966820-da8b-4ece-871b-fbff0f3312c1, 10, Finished, Available, Finished, False)

Keyword types:


SynapseWidget(Synapse.DataFrame, 689f28c4-7f30-438e-8776-f6128ad6dbd4)

Search intent:


SynapseWidget(Synapse.DataFrame, 736c60cb-add6-442c-a0b0-31e801d96b32)

Active flag:


SynapseWidget(Synapse.DataFrame, cb9566bc-d5e5-4f8b-952a-ab9d38db2ae7)

In [9]:
(
    df_keyword_dim.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dim_keyword")
)

print("Silver dim_keyword table created successfully.")

StatementMeta(, b8966820-da8b-4ece-871b-fbff0f3312c1, 11, Finished, Available, Finished, False)

Silver dim_keyword table created successfully.


In [10]:
df_silver_keyword_dim = spark.read.table("dim_keyword")

print(
    "Silver dim_keyword rows:",
    df_silver_keyword_dim.count()
)

print(
    "Silver dim_keyword columns:",
    len(df_silver_keyword_dim.columns)
)

display(df_silver_keyword_dim.limit(10))

StatementMeta(, b8966820-da8b-4ece-871b-fbff0f3312c1, 12, Finished, Available, Finished, False)

Silver dim_keyword rows: 2000
Silver dim_keyword columns: 8


SynapseWidget(Synapse.DataFrame, 8959a01a-40db-43b4-b23c-173225bed3f0)